# Évaluation v3_clean

Compare les différentes configs du pipeline `v3_clean` sur le goldset :

Les résultats sont sauvegardés dans `goldset_runs` pour comparaison dans `08_Eval_Comparison.py`.

## Workflow
1. Lancer ce notebook pour **chaque** config (modifier `CONFIG_TO_RUN`)
2. Lancer `eval_v3clean_metrics.ipynb` pour calculer les métriques RAGAS
3. Visualiser dans `08_Eval_Comparison.py`

In [ ]:
import os, sys, json, time
import pandas as pd
import psycopg
from psycopg.rows import dict_row
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.rag_v3_clean import create_pipeline, RAGConfig
from src.rag_v3_clean.config import (
    RetrievalConfig, SectionAggregationConfig, SelectorConfig,
    ContextBuildConfig, GenerationConfig, QueryProcessorConfig,
    SearchMode, EmbeddingModel, LLMProvider, ContextMode,
)

print("Imports v3_clean OK")

In [ ]:
DSN = os.getenv("TUNNEL_DSN") or os.getenv("SCALINGO_POSTGRESQL_URL") or os.getenv("PG_DSN")

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    cnt = conn.execute("SELECT COUNT(*) as cnt FROM goldset_runs").fetchone()["cnt"]
print(f"DB OK — {cnt} runs existants dans goldset_runs")

## 1. Définition des 3 configurations

Toutes basées sur la config PROD actuelle, seul le retrieval change.

In [ ]:
def make_config(
    tables: list[str],
    section_rerank_top_k: int = 10,
    selector_enabled: bool = True,
    context_mode: ContextMode = ContextMode.WIDE,
) -> RAGConfig:
    """Build a RAGConfig matching prod defaults, only varying retrieval tables."""
    return RAGConfig(
        retrieval=RetrievalConfig(
            tables=tables,
            initial_top_k=20,
            search_mode=SearchMode.SEMANTIC,
            embedding_model=EmbeddingModel.ALBERT,
            enable_chunk_reranker=False,
        ),
        aggregation=SectionAggregationConfig(
            enable_section_reranker=True,
            section_rerank_top_k=section_rerank_top_k,
        ),
        selector=SelectorConfig(enabled=selector_enabled, model="openweight-large"),
        context=ContextBuildConfig(context_mode=context_mode),
        generation=GenerationConfig(model="openweight-large"),
        query_processor=QueryProcessorConfig(
            enable_acronym_expansion=True,
            enable_intent_gating=False,
        ),
        verbose=False,
    )


CONFIGS = {
    "v3clean_prod": make_config(
        tables=["matte", "service_public", "dgafp", "rgrh"],
    ),
    "v3clean_prod_wide_noselector": make_config(
        tables=["matte", "service_public", "dgafp", "rgrh"],
        context_mode=ContextMode.WIDE,
        section_rerank_top_k=5,
        selector_enabled=False,
    ),
    "v3clean_prod_wide": make_config(
        tables=["matte", "service_public", "dgafp", "rgrh"],
        context_mode=ContextMode.WIDE,
        selector_enabled=True,
    ),
}


def load_prod_config() -> RAGConfig:
    """Build a RAGConfig from the live Admin Config DB — exact mirror of 01_Chatbot.py."""
    from src.rag_v3_clean.admin import get_rag_config
    from src.rag_v3_clean import get_default_config

    rc = get_rag_config()
    cfg = get_default_config()

    _mode_map = {"standard": ContextMode.STANDARD, "wide": ContextMode.WIDE}
    cfg.context.context_mode = _mode_map.get(rc.v3_context_mode, ContextMode.STANDARD)
    cfg.context.token_budget = rc.v3_token_budget
    cfg.context.doc_entire_threshold = rc.v3_doc_entire_threshold

    cfg.selector.enabled = rc.v3_enable_selector
    cfg.selector.model = rc.v3_selector_model
    cfg.selector.prompt_name = rc.v3_selector_prompt_name

    cfg.query_processor.enable_intent_gating = rc.enable_intent_gating
    cfg.query_processor.enable_acronym_expansion = rc.enable_query_expansion
    cfg.query_processor.intent_prompt_name = rc.v3_intent_prompt_name

    cfg.retrieval.tables = rc.v3_tables
    cfg.retrieval.initial_top_k = rc.v3_initial_top_k
    cfg.retrieval.alpha = rc.v3_alpha
    _sm = {"semantic": SearchMode.SEMANTIC, "hybrid": SearchMode.HYBRID, "lexical": SearchMode.LEXICAL}
    cfg.retrieval.search_mode = _sm.get(rc.v3_search_mode, SearchMode.SEMANTIC)

    cfg.aggregation.enable_section_reranker = rc.v3_enable_reranker
    cfg.aggregation.section_rerank_top_k = rc.v3_rerank_top_k

    cfg.generation.model = rc.v3_generator_model
    cfg.generation.temperature = rc.v3_temperature
    cfg.generation.system_prompt_name = rc.v3_system_prompt_name

    cfg.verbose = False
    return cfg


CONFIGS["v3clean_prod_full"] = load_prod_config()
print(f"\n  v3clean_prod_full: LIVE from Admin Config DB")
_pc = CONFIGS["v3clean_prod_full"]
print(f"    tables={_pc.retrieval.tables}")
print(f"    selector={_pc.selector.enabled}, intent_gating={_pc.query_processor.enable_intent_gating}")
print(f"    context_mode={_pc.context.context_mode.value}, rerank_top_k={_pc.aggregation.section_rerank_top_k}")

for name, cfg in CONFIGS.items():
    tables_str = ", ".join(cfg.retrieval.tables)
    print(f"  {name}: [{tables_str}]")


## 2. Chargement du goldset

In [ ]:
GOLDSET_FILTER = {
    "has_gold_answer": True,
    "tags": ["v3clean_b"],
    "goldset_name": "synthetic_docs_v1",
    "limit": None,
}


def load_questions(filters: dict) -> list[dict]:
    where = ["gold_sources IS NOT NULL", "gold_sources != ''"]
    if filters.get("has_gold_answer"):
        where.append("gold_answer IS NOT NULL AND gold_answer != ''")
    if filters.get("tags"):
        tag_arr = "ARRAY[" + ",".join(f"'{t}'" for t in filters["tags"]) + "]"
        where.append(f"tags @> {tag_arr}")
    if filters.get("goldset_name"):
        where.append(f"goldset_name = '{filters['goldset_name']}'")

    limit = f"LIMIT {filters['limit']}" if filters.get("limit") else ""
    sql = f"""
        SELECT id, question, gold_answer, gold_sources, theme, tags, goldset_name
        FROM goldset_questions_v2
        WHERE {' AND '.join(where)}
        ORDER BY id
        {limit}
    """
    with psycopg.connect(DSN, row_factory=dict_row) as conn:
        return conn.execute(sql).fetchall()


questions = load_questions(GOLDSET_FILTER)
print(f"{len(questions)} questions chargées")
if questions:
    print(f"  Exemple: {questions[0]['question'][:80]}...")
    df_q = pd.DataFrame(questions)
    print(f"  Répartition goldsets: {dict(df_q['goldset_name'].value_counts())}")

## 3. Sélection de la config à lancer

Modifier `CONFIG_TO_RUN` pour chaque exécution.

In [ ]:
CONFIG_TO_RUN = "v3clean_prod_full"  # <<< Config prod complète (tables + selector + intent gater)

config = CONFIGS[CONFIG_TO_RUN]
print(f"Config: {CONFIG_TO_RUN}")
print(f"  Tables: {config.retrieval.tables}")
print(f"  Selector: {config.selector.enabled}")
print(f"  Rerank top_k: {config.aggregation.section_rerank_top_k}")
print(f"  Questions: {len(questions)}")


In [ ]:
pipeline = create_pipeline(config, dsn=DSN)
print("Pipeline v3_clean created")

## 4. Fonctions utilitaires

In [ ]:
MAX_RETRIES = 3
DELAY_BETWEEN_QUESTIONS = 0.5


def get_already_run_ids(config_name: str) -> set:
    """Return question_ids already present for this config."""
    with psycopg.connect(DSN, row_factory=dict_row) as conn:
        rows = conn.execute(
            "SELECT DISTINCT question_id FROM goldset_runs WHERE config_name = %s",
            (config_name,),
        ).fetchall()
    return {r["question_id"] for r in rows}


def format_context_for_db(context_items) -> list[dict]:
    """Convert PipelineResult.context_items to the JSONB format expected by goldset_runs."""
    out = []
    for item in context_items:
        out.append({
            "source": item.publisher or "unknown",
            "title": item.document_title or "",
            "section_heading": item.heading or "",
            "text": item.content or "",
            "score": item.score,
            "token_count": item.token_estimate,
            "doc_url": item.document_url or "",
            "section_id": item.section_id,
        })
    return out


def save_goldset_run(
    question_id: int,
    config_name: str,
    config_params: dict,
    response: str,
    context_items_json: list,
    retrieval_time_ms: int | None = None,
    generation_time_ms: int | None = None,
) -> int:
    """Insert one generation result into goldset_runs. Returns the run id."""
    sql = """
        INSERT INTO goldset_runs (
            question_id, config_name, config_params, response,
            retrieved_context, llm_model, embedding_model,
            retrieval_time_ms, generation_time_ms
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        RETURNING id
    """
    with psycopg.connect(DSN) as conn:
        row = conn.execute(sql, (
            question_id,
            config_name,
            json.dumps(config_params, ensure_ascii=False),
            response,
            json.dumps(context_items_json, ensure_ascii=False),
            config_params.get("generator_model", "openweight-large"),
            config_params.get("embedding_model", "albert"),
            retrieval_time_ms,
            generation_time_ms,
        )).fetchone()
        conn.commit()
    return row[0]

## 5. Test rapide (10 questions)

Toujours tester sur un petit batch avant de lancer le run complet.

In [ ]:
## Test sur 1 question d'abord
test_q = questions[0]
print(f"Test: {test_q['question'][:80]}...")

result = pipeline.run(test_q["question"])
print(f"  Answer: {result.answer[:200]}...")
print(f"  Context items: {len(result.context_items)}")
print(f"  Timing: {result.timing}")
for ci in result.context_items[:3]:
    has_text = "✅" if ci.content and len(ci.content) > 10 else "❌"
    print(f"  [{ci.publisher}] {ci.heading[:50]} | score={ci.score:.3f} | text={has_text} ({ci.token_estimate} tok)")

In [ ]:
## Test batch sur 10 questions (pas sauvegardé en DB)
TEST_N = 10
test_results = []

for i, q in enumerate(questions[:TEST_N]):
    t0 = time.time()
    try:
        result = pipeline.run(q["question"])
        elapsed = time.time() - t0
        ctx_json = format_context_for_db(result.context_items)
        ret_ms = int(result.timing.get("retrieval_ms", 0))
        gen_ms = int(result.timing.get("generation_ms", 0))

        test_results.append({
            "question_id": q["id"],
            "question": q["question"][:60],
            "success": True,
            "n_items": len(result.context_items),
            "ret_ms": ret_ms,
            "gen_ms": gen_ms,
            "total_ms": int(elapsed * 1000),
            "answer_len": len(result.answer),
        })
        print(f"[{i+1}/{TEST_N}] Q{q['id']}: ✅ {len(result.context_items)} items, "
              f"ret={ret_ms}ms gen={gen_ms}ms total={int(elapsed*1000)}ms | {result.answer[:80]}...")
    except Exception as e:
        elapsed = time.time() - t0
        test_results.append({
            "question_id": q["id"],
            "question": q["question"][:60],
            "success": False,
            "n_items": 0, "ret_ms": 0, "gen_ms": 0,
            "total_ms": int(elapsed * 1000),
            "answer_len": 0,
        })
        print(f"[{i+1}/{TEST_N}] Q{q['id']}: ❌ {e}")

df_test = pd.DataFrame(test_results)
print(f"\n{'='*60}")
print(f"Résumé: {df_test['success'].sum()}/{TEST_N} OK")
print(f"  Avg items: {df_test['n_items'].mean():.1f}")
print(f"  Avg retrieval: {df_test['ret_ms'].mean():.0f}ms")
print(f"  Avg generation: {df_test['gen_ms'].mean():.0f}ms")
print(f"  Avg total: {df_test['total_ms'].mean():.0f}ms")
print(f"  Avg answer length: {df_test['answer_len'].mean():.0f} chars")
display(df_test)

## 5. Boucle de génération

In [ ]:
# ━━━ CONTRÔLE DU BATCH ━━━
FRESH_START = True    # True = purge les runs existants pour cette config et repart de 0
BATCH_LIMIT = None    # None = toutes les questions du tag v3clean_b (113)

# Purge si demandé
if FRESH_START:
    with psycopg.connect(DSN) as conn:
        deleted = conn.execute(
            "DELETE FROM goldset_runs WHERE config_name = %s", (CONFIG_TO_RUN,)
        ).rowcount
        conn.commit()
    print(f"FRESH START: {deleted} runs supprimés pour '{CONFIG_TO_RUN}'")

# Skip questions already in DB for this config
already_run = get_already_run_ids(CONFIG_TO_RUN)
to_run = [q for q in questions if q["id"] not in already_run]

if BATCH_LIMIT:
    to_run = to_run[:BATCH_LIMIT]

print(f"Already run: {len(already_run)} | To run: {len(to_run)}"
      + (f" (limité à {BATCH_LIMIT})" if BATCH_LIMIT else " (toutes)"))

In [ ]:
config_params = {
    "config_mode": "v3_clean",
    "tables": config.retrieval.tables,
    "search_mode": config.retrieval.search_mode.value,
    "embedding_model": config.retrieval.embedding_model.value,
    "initial_top_k": config.retrieval.initial_top_k,
    "token_budget": config.context.token_budget,
    "selector_enabled": config.selector.enabled,
    "selector_model": config.selector.model,
    "generator_model": config.generation.model,
    "temperature": config.generation.temperature,
    "system_prompt": config.generation.system_prompt_name,
    "section_reranker": config.aggregation.enable_section_reranker,
    "section_rerank_top_k": config.aggregation.section_rerank_top_k,
}

results = []
errors = []
successes = 0
total = len(to_run)

print(f"Starting batch: {total} questions with config '{CONFIG_TO_RUN}'")
print(f"  Save to DB: True | Skip existing: True")
print()

t_start = time.time()
for i, q in enumerate(to_run):
    qid = q["id"]
    question_short = q["question"][:60]

    for attempt in range(MAX_RETRIES):
        try:
            t0 = time.time()
            result = pipeline.run(q["question"])
            elapsed_q = time.time() - t0

            ctx_json = format_context_for_db(result.context_items)
            retrieval_ms = int(result.timing.get("retrieval_ms", 0))
            generation_ms = int(result.timing.get("generation_ms", 0))

            run_id = save_goldset_run(
                question_id=qid,
                config_name=CONFIG_TO_RUN,
                config_params=config_params,
                response=result.answer,
                context_items_json=ctx_json,
                retrieval_time_ms=retrieval_ms,
                generation_time_ms=generation_ms,
            )

            successes += 1
            results.append({
                "question_id": qid,
                "question": q["question"],
                "gold_answer": q.get("gold_answer", ""),
                "response": result.answer,
                "n_context_items": len(result.context_items),
                "retrieval_ms": retrieval_ms,
                "generation_ms": generation_ms,
                "elapsed_ms": int(elapsed_q * 1000),
                "run_id": run_id,
            })
            print(f"[{i+1}/{total}] Q{qid}: {len(result.context_items)} items, "
                  f"ret={retrieval_ms}ms gen={generation_ms}ms total={int(elapsed_q*1000)}ms "
                  f"→ run #{run_id}")
            break

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait = 2 ** attempt
                print(f"[{i+1}/{total}] Q{qid}: attempt {attempt+1} failed: {str(e)[:80]}... retry in {wait}s")
                time.sleep(wait)
            else:
                print(f"[{i+1}/{total}] Q{qid}: FAILED after {MAX_RETRIES} attempts: {str(e)[:100]}")
                errors.append({"question_id": qid, "question": q["question"], "error": str(e)})

    time.sleep(DELAY_BETWEEN_QUESTIONS)

    if (i + 1) % 25 == 0:
        elapsed = time.time() - t_start
        rate = (i + 1) / elapsed
        eta = (total - i - 1) / rate
        print(f"--- Progress: {i+1}/{total} ({successes} OK, {len(errors)} err) | "
              f"{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining ---")

elapsed_total = time.time() - t_start
print(f"\n{'='*60}")
print(f"Batch complete in {elapsed_total:.0f}s")
print(f"  OK: {successes}/{total}")
print(f"  Failed: {len(errors)}/{total}")
if results:
    df_results = pd.DataFrame(results)
    print(f"  Avg retrieval: {df_results['retrieval_ms'].mean():.0f}ms")
    print(f"  Avg generation: {df_results['generation_ms'].mean():.0f}ms")
    print(f"  Avg total: {df_results['elapsed_ms'].mean():.0f}ms")
    print(f"  Avg context items: {df_results['n_context_items'].mean():.1f}")
    print(f"  Avg answer length: {df_results['response'].str.len().mean():.0f} chars")


## 6. Export CSV (backup)

In [ ]:
df = pd.DataFrame(results)
csv_path = f"gen_{CONFIG_TO_RUN}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df.to_csv(csv_path, index=False)
print(f"CSV: {csv_path}")
print(f"  {len(df)} runs, avg {df['n_context_items'].mean():.1f} items, avg {df['response'].str.len().mean():.0f} chars")

if errors:
    print(f"\nErrors ({len(errors)}):")
    for e in errors[:5]:
        print(f"  Q{e['question_id']}: {e['error'][:100]}")

## 7. Retry des questions en échec

In [ ]:
if errors:
    print(f"Retrying {len(errors)} failed questions...")
    retry_successes = 0
    retry_still_failed = []

    for err in errors:
        qid = err["question_id"]
        question = err["question"]
        for attempt in range(MAX_RETRIES):
            try:
                t0 = time.time()
                result = pipeline.run(question)
                elapsed_q = time.time() - t0
                ctx_json = format_context_for_db(result.context_items)
                retrieval_ms = int(result.timing.get("retrieval_ms", 0))
                generation_ms = int(result.timing.get("generation_ms", 0))

                run_id = save_goldset_run(
                    question_id=qid,
                    config_name=CONFIG_TO_RUN,
                    config_params=config_params,
                    response=result.answer,
                    context_items_json=ctx_json,
                    retrieval_time_ms=retrieval_ms,
                    generation_time_ms=generation_ms,
                )
                retry_successes += 1
                results.append({
                    "question_id": qid, "question": question,
                    "gold_answer": "", "response": result.answer,
                    "n_context_items": len(result.context_items),
                    "retrieval_ms": retrieval_ms, "generation_ms": generation_ms,
                    "elapsed_ms": int(elapsed_q * 1000), "run_id": run_id,
                })
                print(f"  Q{qid}: OK on retry → run #{run_id}")
                break
            except Exception as e:
                if attempt == MAX_RETRIES - 1:
                    retry_still_failed.append({"question_id": qid, "error": str(e)})
                    print(f"  Q{qid}: still failing: {str(e)[:80]}")
                else:
                    time.sleep(2 ** attempt)
        time.sleep(DELAY_BETWEEN_QUESTIONS)

    print(f"\nRetry results: {retry_successes} recovered, {len(retry_still_failed)} still failing")
    errors = retry_still_failed
else:
    print("No errors to retry")

## 8. Vérification rapide

In [ ]:
import random

sample = random.sample(results, min(5, len(results)))
for r in sample:
    print(f"\nQ: {r['question'][:100]}")
    print(f"A: {r['response'][:200]}...")
    print(f"   Items: {r['n_context_items']}, Retrieval: {r['retrieval_ms']}ms, Gen: {r['generation_ms']}ms")

## 9. Diagnostic: qualité du retrieved_context

Vérifie que toutes les runs ont un contexte exploitable pour les métriques RAGAS.

In [ ]:
from collections import defaultdict

diag_query = """
    SELECT gr.id as run_id, gr.config_name, gr.question_id,
           gq.question, gr.retrieved_context
    FROM goldset_runs gr
    JOIN goldset_questions_v2 gq ON gr.question_id = gq.id
    WHERE gr.config_name LIKE 'v3clean_%'
    ORDER BY gr.config_name, gr.id
"""

with psycopg.connect(DSN, row_factory=dict_row) as conn:
    all_runs = conn.execute(diag_query).fetchall()

print(f"Total v3clean runs in DB: {len(all_runs)}\n")

stats = defaultdict(lambda: {"total": 0, "good": 0, "empty": 0, "no_text": 0, "avg_items": []})

for run in all_runs:
    cfg = run["config_name"]
    s = stats[cfg]
    s["total"] += 1

    ctx = run["retrieved_context"]
    if isinstance(ctx, str):
        try:
            ctx = json.loads(ctx)
        except Exception:
            ctx = None

    if not ctx:
        s["empty"] += 1
        continue

    s["avg_items"].append(len(ctx))
    has_text = any(item.get("text") and len(item.get("text", "")) > 10 for item in ctx)
    if has_text:
        s["good"] += 1
    else:
        s["no_text"] += 1

for cfg, s in sorted(stats.items()):
    avg_items = sum(s["avg_items"]) / len(s["avg_items"]) if s["avg_items"] else 0
    pct = s["good"] / s["total"] * 100 if s["total"] > 0 else 0
    verdict = "ALL GOOD" if s["good"] == s["total"] else f"{pct:.0f}% usable"
    print(f"{cfg}: {s['total']} runs | {s['good']} good, {s['empty']} empty, {s['no_text']} no-text | "
          f"avg {avg_items:.1f} items | {verdict}")

In [ ]:
# Distribution dans la DB
with psycopg.connect(DSN, row_factory=dict_row) as conn:
    rows = conn.execute("""
        SELECT config_name, COUNT(*) as cnt, 
               AVG(retrieval_time_ms) as avg_ret, AVG(generation_time_ms) as avg_gen
        FROM goldset_runs 
        WHERE config_name LIKE 'v3clean_%'
        GROUP BY config_name ORDER BY config_name
    """).fetchall()

print("Runs v3clean_* dans goldset_runs:")
for r in rows:
    print(f"  {r['config_name']}: {r['cnt']} runs, avg ret={r['avg_ret']:.0f}ms, gen={r['avg_gen']:.0f}ms")

---

**Next step:** Lancer `eval_v3clean_metrics.ipynb` pour calculer les métriques RAGAS sur ces runs.